# DAWIN Quickstart (30-Second PyPI Demo)
Installs `dawin` from PyPI without cloning the repository.


In [ ]:
!pip install -q --upgrade dawin
import time, torch, torch.nn as nn, matplotlib.pyplot as plt, dawin
from dawin import (
  DAWINLinear, TritonDAWINLinear, DAWINMLP,
  DAWINTransformer, DAWINGNN, compute_dawin_laplacian_loss,
)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('dawin', dawin.__version__, '| PyTorch', torch.__version__, '|', dev)
x, dt = torch.randn(4, 64, 16, device=dev), torch.rand(4, 64, device=dev)*0.5
m1 = DAWINLinear(16, 32, mode='quad').to(dev)
m2 = TritonDAWINLinear(16, 32, mode='quad').to(dev)
m2.load_state_dict(m1.state_dict())
print('Scan vs Triton max diff:', f'{(m1(x,dt)-m2(x,dt)).abs().max():.2e}')


In [ ]:
torch.manual_seed(42)
def make_osc(n, L, a, b):
  dt = torch.empty(n, L).uniform_(a, b); dt[:, 0] = 0.0
  t = torch.cumsum(dt, 1)
  amp = torch.empty(n, 1).uniform_(0.6, 1.4)
  ph = torch.empty(n, 1).uniform_(0.0, 6.28318)
  s = torch.stack([amp*torch.exp(-0.35*t)*torch.cos(3.2*t+ph),
                   amp*torch.exp(-0.35*t)*torch.sin(3.2*t+ph)], -1)
  return s[:, :-1].to(dev), dt[:, 1:].to(dev), s[:, 1:].to(dev)
xt, dtt, yt = make_osc(512, 64, 0.02, 0.45)
xe, dte, ye = make_osc(128, 64, 0.45, 1.10)
dm = DAWINMLP(2, 2, 48, 2, use_triton=torch.cuda.is_available()).to(dev)
rm = nn.Sequential(nn.Linear(3,104), nn.ReLU(), nn.Linear(104,104),
                   nn.ReLU(), nn.Linear(104,2)).to(dev)
od, or_ = torch.optim.AdamW(dm.parameters(), 3e-3), torch.optim.AdamW(rm.parameters(), 3e-3)
t0 = time.time()
for _ in range(180):
  od.zero_grad()
  ld = nn.functional.mse_loss(dm(xt, dtt), yt) + 1e-4*compute_dawin_laplacian_loss(dm)
  ld.backward(); od.step()
  or_.zero_grad()
  lr = nn.functional.mse_loss(rm(torch.cat([xt, dtt.unsqueeze(-1)], -1)), yt)
  lr.backward(); or_.step()
print(f'Done in {time.time()-t0:.1f}s | Train MSE DAWIN={ld:.5f} ReLU={lr:.5f}')


In [ ]:
dm.eval(); rm.eval()
with torch.no_grad():
  pd, pr = dm(xe, dte), rm(torch.cat([xe, dte.unsqueeze(-1)], -1))
  md, mr = nn.functional.mse_loss(pd, ye).item(), nn.functional.mse_loss(pr, ye).item()
  ms = nn.functional.mse_loss(dm(xe, dte[torch.randperm(len(dte), device=dev)]), ye).item()
print(f'Extrap MSE: DAWIN={md:.5f} vs ReLU={mr:.5f} | Shuffled dt={ms:.5f}')
ta = torch.cumsum(dte[0].cpu(), 0).numpy()
plt.figure(figsize=(8, 3))
plt.plot(ta, ye[0,:,0].cpu(), 'k-o', ms=3, label='Truth')
plt.plot(ta, pd[0,:,0].cpu(), '#b31b1b', lw=2, label=f'DAWIN ({md:.4f})')
plt.plot(ta, pr[0,:,0].cpu(), '#4a6fe3', ls='--', label=f'ReLU ({mr:.4f})')
plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.show()


In [ ]:
tf = DAWINTransformer(8, 4, 32, 4, 2).to(dev)
ytf = tf(torch.randn(2, 32, 8, device=dev), torch.rand(2, 32, device=dev)*0.5)
gnn = DAWINGNN(6, 3, 32, 2).to(dev)
adj = torch.eye(6, device=dev).unsqueeze(0).expand(2, -1, -1)
ygnn = gnn(torch.randn(2,24,6,6,device=dev), torch.rand(2,24,6,device=dev)*0.5, adj)
print('DAWINTransformer:', tuple(ytf.shape), '| DAWINGNN:', tuple(ygnn.shape))
